## Unit 2: Web Application Development with Flask & Databases
### 📘 Companion Guide: Direct SQLite Integration in Flask (Without ORM)


### 🎯 Key Learning Objectives
1. **The Request Lifecycle & `g`**: Store database connections safely per request using Flask's `g` proxy.
2. **Preventing Connection Leaks**: Automatically close connections using `@app.teardown_appcontext`.
3. **The `sqlite3.Row` Row Factory**: Access columns by name (`row['name']`) instead of numeric indices.
4. **Parameterized Queries**: Understand the `?` placeholder syntax and why direct string formatting causes SQL Injection.
5. **Direct CRUD**: Implement Create, Read, Update, and Delete routes using raw SQL statements.


In [ ]:
# Cell 1: Environment Verification
import sys
import sqlite3
import flask

print("=" * 60)
print(f"🐍 Python Version : {sys.version.split()[0]}")
print(f"🌶️ Flask Version  : {flask.__version__}")
print(f"💾 SQLite Version : {sqlite3.sqlite_version}")
print("=" * 60)
print("✅ Native SQLite & Flask environment ready!")


---
## 🌐 1. Managing Database Connections in Flask: The `g` Object

In web applications, **never** open a single global connection at the top of your file:
```python
# ❌ NEVER DO THIS IN FLASK:
db = sqlite3.connect("database.db")  # Crashes when multiple users connect simultaneously!
```

### Why?
Web servers handle requests across multiple threads or processes. A single connection shared across threads causes race conditions, transaction conflicts, and socket crashes.

### The Correct Flask Pattern:
1. **Open a connection on demand** for the current request using Flask's `g` object.
2. **Store it in `g._database`** so multiple functions in the same request share it.
3. **Close it automatically** when the request ends using `@app.teardown_appcontext`.

Let's see this pattern in action:


In [ ]:
# Cell 2: The Flask 'g' Connection Helper & Teardown Hook
from flask import Flask, g, jsonify, request
import sqlite3

raw_app = Flask("raw_sqlite_app")
raw_app.config["TESTING"] = True

# We will use an in-memory SQLite database for fast, isolated classroom testing
DATABASE = ':memory:'

def get_db():
    """
    Returns the database connection for the current request context.
    Creates a new connection if one does not already exist.
    """
    db = getattr(g, '_database', None)
    if db is None:
        db = g._database = sqlite3.connect(DATABASE)
        # Enable column access by name (e.g., row['name'] instead of row[0])
        db.row_factory = sqlite3.Row
    return db

@raw_app.teardown_appcontext
def close_connection(exception):
    """
    Flask automatically calls this function after every request finishes.
    This guarantees no database connections are leaked!
    """
    db = getattr(g, '_database', None)
    if db is not None:
        db.close()

print("✅ Cell 2 Complete: Connection manager and teardown hook configured!")


---
## 🔨 2. Creating Tables with Raw SQL (`CREATE TABLE`)

To create a table, we write standard SQL `CREATE TABLE` and execute it through our database cursor.

Notice the SQLite data types:
- `INTEGER PRIMARY KEY AUTOINCREMENT`: Unique auto-incrementing identifier.
- `TEXT NOT NULL`: String that cannot be empty.
- `REAL`: Floating-point number (e.g., GPA).

Let's create a `students` table:


In [ ]:
# Cell 3: Create the 'students' table
def init_db():
    """Creates tables if they do not already exist."""
    with raw_app.app_context():
        db = get_db()
        cursor = db.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS students (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                course TEXT NOT NULL,
                gpa REAL DEFAULT 0.0
            );
        ''')
        db.commit()
        print("✅ Cell 3 Complete: 'students' table created successfully via Raw SQL!")

init_db()


---
## ➕ 3. CREATE: Inserting Data with Parameterized Queries (`?`)

> 🚨 **CRITICAL SECURITY RULE FOR MSc IT STUDENTS:**  
> **NEVER concatenate user input into SQL queries!**  
> ❌ `cursor.execute(f"INSERT INTO students VALUES ('{name}')")`  $\longleftarrow$ **Vulnerable to SQL Injection!**  
> ✅ `cursor.execute("INSERT INTO students VALUES (?)", (name,))` $\longleftarrow$ **Safe & Parameterized!**

With `?` placeholders:
- SQLite treats the value strictly as data, never as executable SQL code.
- SQLite handles escaping and type casting automatically.

Let's insert 3 student records:


In [ ]:
# Cell 4: Insert student records using parameterized queries (?)
with raw_app.app_context():
    db = get_db()
    cursor = db.cursor()

    # Sample student data (tuples)
    students_data = [
        ("Aarav Sharma", "aarav@uni.edu", "MSc IT", 3.85),
        ("Diya Patel", "diya@uni.edu", "MSc IT", 3.92),
        ("Rohan Verma", "rohan@uni.edu", "MSc CS", 3.35)
    ]

    # Execute parameterized INSERT query for each student
    for s in students_data:
        cursor.execute('''
            INSERT INTO students (name, email, course, gpa)
            VALUES (?, ?, ?, ?)
        ''', s)

    # Commit changes to database!
    db.commit()

    print(f"✅ Cell 4 Complete: Inserted {len(students_data)} records! Last ID: {cursor.lastrowid}")


---
## 📋 4. READ: Querying Data with the `sqlite3.Row` Row Factory

By default, Python's `sqlite3` returns plain tuples:
```python
row = cursor.fetchone()
print(row[1])  # What column is index 1? Confusing and error-prone!
```

Because we set `db.row_factory = sqlite3.Row` in `get_db()`, we can access columns **by column name**:
```python
print(row['name'])   # Clear and readable!
print(row['email'])  # Self-documenting!
```

Let's query and display all students:


In [ ]:
# Cell 5: Query and display all records
with raw_app.app_context():
    db = get_db()
    cursor = db.cursor()

    # Query all students
    cursor.execute("SELECT id, name, email, course, gpa FROM students ORDER BY gpa DESC")
    rows = cursor.fetchall()

    print(f"Total students retrieved: {len(rows)}\n")
    print(f"{'ID':<4} | {'Name':<15} | {'Course':<10} | {'GPA':<5} | {'Email'}")
    print("-" * 55)
    for r in rows:
        # Accessing by column name using sqlite3.Row
        print(f"{r['id']:<4} | {r['name']:<15} | {r['course']:<10} | {r['gpa']:<5.2f} | {r['email']}")

print("\n✅ Cell 5 Complete: All rows queried using sqlite3.Row!")


---
## ✏️ 5. UPDATE and DELETE with Raw SQL

### Updating Records:
```sql
UPDATE students SET gpa = ? WHERE id = ?
```

### Deleting Records:
```sql
DELETE FROM students WHERE id = ?
```

Let's execute both and check `cursor.rowcount` (which tells you how many rows were affected):


In [ ]:
# Cell 6: Raw SQL UPDATE and DELETE operations
with raw_app.app_context():
    db = get_db()
    cursor = db.cursor()

    # --- 1. UPDATE ---
    # Update Rohan's GPA (ID #3)
    cursor.execute("UPDATE students SET gpa = ? WHERE id = ?", (3.70, 3))
    db.commit()
    print(f"UPDATE executed: {cursor.rowcount} row(s) affected.")

    # --- 2. DELETE ---
    # Delete student with ID #1
    cursor.execute("DELETE FROM students WHERE id = ?", (1,))
    db.commit()
    print(f"DELETE executed: {cursor.rowcount} row(s) affected.")

    # --- 3. Verify Remaining Records ---
    cursor.execute("SELECT id, name, gpa FROM students")
    remaining = cursor.fetchall()
    print("\nRemaining Students:")
    for r in remaining:
        print(f"  - #{r['id']}: {r['name']} (GPA: {r['gpa']})")

print("\n✅ Cell 6 Complete: UPDATE and DELETE verified!")


---
## 🌐 6. Building Web Routes with Raw SQLite & `app.test_client()`

Now let's wire our raw SQL functions into Flask routes:
- `GET /api/students`: Return JSON list of students.
- `POST /api/students`: Insert student from incoming JSON.
- `GET /api/students/<int:id>`: Fetch single student by primary key.

Let's register the routes and test them:


In [ ]:
# Cell 7: Register Flask Web Routes using Raw SQL
@raw_app.route('/api/students', methods=['GET'])
def get_students_route():
    db = get_db()
    cursor = db.cursor()
    cursor.execute("SELECT id, name, email, course, gpa FROM students")
    rows = cursor.fetchall()

    # Convert sqlite3.Row objects into standard JSON-serializable dictionaries
    data = [dict(row) for row in rows]
    return jsonify(data), 200


@raw_app.route('/api/students', methods=['POST'])
def add_student_route():
    data = request.get_json()
    if not data or not data.get('name') or not data.get('email') or not data.get('course'):
        return jsonify({"error": "Missing required fields"}), 400

    db = get_db()
    cursor = db.cursor()
    try:
        cursor.execute('''
            INSERT INTO students (name, email, course, gpa)
            VALUES (?, ?, ?, ?)
        ''', (data['name'], data['email'], data['course'], float(data.get('gpa', 0.0))))
        db.commit()
        return jsonify({"message": "Student created", "id": cursor.lastrowid}), 201
    except sqlite3.IntegrityError:
        return jsonify({"error": f"Email '{data['email']}' is already in use"}), 409


@raw_app.route('/api/students/<int:student_id>', methods=['GET'])
def get_student_by_id_route(student_id):
    db = get_db()
    cursor = db.cursor()
    cursor.execute("SELECT id, name, email, course, gpa FROM students WHERE id = ?", (student_id,))
    row = cursor.fetchone()
    if row is None:
        return jsonify({"error": "Student not found"}), 404
    return jsonify(dict(row)), 200

# Test all routes using Flask's test_client
with raw_app.test_client() as client:
    print("--- 1. Testing GET /api/students ---")
    get_res = client.get('/api/students')
    print(f"Status: {get_res.status_code} | Data: {get_res.get_json()}")

    print("\n--- 2. Testing POST /api/students ---")
    post_res = client.post('/api/students', json={
        "name": "Kavya Menon",
        "email": "kavya@uni.edu",
        "course": "MSc AI",
        "gpa": 3.95
    })
    print(f"Status: {post_res.status_code} | Created: {post_res.get_json()}")

    print("\n--- 3. Testing GET /api/students/<id> ---")
    new_id = post_res.get_json()["id"]
    single_res = client.get(f'/api/students/{new_id}')
    print(f"Status: {single_res.status_code} | Fetched: {single_res.get_json()}")

print("\n✅ Cell 7 Complete: Full Raw SQL Web Service Operational!")


---
## 🛡️ 7. Live Demonstration: Why Parameterized Queries Prevent SQL Injection

Let's see what happens when insecure string formatting is used vs parameterized queries:

### Insecure Query:
```python
f"SELECT * FROM users WHERE username = '{user_input}'"
```
If an attacker enters:
```
' OR 1=1 --
```
The query becomes:
```sql
SELECT * FROM users WHERE username = '' OR 1=1 --'
```
Because `1=1` is always True, the database dumps **all user passwords and data**!

Let's test this vulnerability live:


In [ ]:
# Cell 8: Security Comparison — Insecure vs Secure Parameterized Query
with raw_app.app_context():
    db = get_db()
    cursor = db.cursor()

    # Attacker's malicious input
    malicious_input = "' OR 1=1 --"

    print("--- Demonstration 1: VULNERABLE Code (String Formatting) ---")
    # ❌ Dangerous query with string concatenation
    insecure_query = f"SELECT id, name FROM students WHERE email = '{malicious_input}'"
    print(f"Generated SQL: {insecure_query}")
    cursor.execute(insecure_query)
    leaked_data = cursor.fetchall()
    print(f"🚨 Attack Successful! Leaked {len(leaked_data)} record(s): {[dict(r) for r in leaked_data]}")

    print("\n--- Demonstration 2: SECURE Code (Parameterized Query with ?) ---")
    # ✅ Secure parameterized query
    secure_query = "SELECT id, name FROM students WHERE email = ?"
    print(f"Generated SQL: {secure_query} with param: {malicious_input}")
    cursor.execute(secure_query, (malicious_input,))
    safe_data = cursor.fetchall()
    print(f"🛡️ Attack Blocked! Records returned: {len(safe_data)} (Expected 0)")

print("\n✅ Cell 8 Complete: SQL Injection defense clearly demonstrated!")


---
## ⚖️ 8. Summary: Raw SQLite vs SQLAlchemy ORM

Now that you have seen both approaches, here is how they compare:

| Feature | Direct SQLite (`sqlite3`) | Flask-SQLAlchemy (ORM) |
|---|---|---|
| **Setup** | Built-in to Python (Zero install) | Requires `Flask-SQLAlchemy` package |
| **Data Representation** | Tuples or `sqlite3.Row` dictionaries | Python Model Objects (`student.name`) |
| **Syntax** | Hand-written SQL strings | Python expressions (`Student.query.filter(...)`) |
| **SQL Injection Safety** | Requires manual `?` discipline | Automatic parameterization by design |
| **Database Portability** | Tied to SQLite dialect | Switch SQLite $\leftrightarrow$ MySQL with 1 line |
| **Relationships (1:N)** | Manual `JOIN` queries | Automatic traversal (`student.department.name`) |
| **Best Used For** | Tiny scripts, microservices, learning SQL | Full-featured, scalable web applications |

---

## 🎯 Next Steps:
Ready to explore declarative models and automated migrations? Open:
- **`03_database_integration.ipynb`**: The complete step-by-step Flask-SQLAlchemy master guide!
